# YTAgent — Free AI 3D Video Worker

This notebook runs Wan 2.1 on Kaggle's free GPU and connects it to your YTAgent Railway app. Turn **Internet on** and select a **GPU** accelerator in Notebook settings before running. Keep this notebook running while a video is being generated.

In [ ]:
import os, subprocess, pathlib
repo_dir = "/kaggle/working/YTAgent"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/qingremzy/YTAgent.git", repo_dir], check=True)
os.chdir(repo_dir + "/gpu-worker")
print("Installing worker packages...")
subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)
cloudflared = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", cloudflared], check=True)
    os.chmod(cloudflared, 0o755)
print("Setup complete. Continue to the next cell.")

In [ ]:
import secrets, os
worker_token = secrets.token_urlsafe(36)
os.environ["YT_AGENT_WORKER_TOKEN"] = worker_token
print("COPY THIS TOKEN INTO RAILWAY AS YT_AGENT_WORKER_TOKEN:\n")
print(worker_token)

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Kaggle Notebook settings, select a GPU accelerator.")

In [ ]:
import subprocess, time, os
worker_log = open("/tmp/ytagent-worker.log", "w")
worker_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd=os.getcwd(), stdout=worker_log, stderr=subprocess.STDOUT
)
time.sleep(5)
if worker_process.poll() is not None:
    print(open("/tmp/ytagent-worker.log").read()[-4000:])
    raise RuntimeError("GPU worker failed to start. Check the error above.")
print("GPU worker is running. Now run the next cell and keep it running.")

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:7860

## Connect YTAgent

When the last cell prints a URL ending in `trycloudflare.com`, copy it into Railway as `VIDEO_WORKER_URL`. Copy the token from the third cell into Railway as `YT_AGENT_WORKER_TOKEN`. Redeploy YTAgent if needed. Do not share the token publicly.

The free GPU notebook must remain active while YTAgent generates scenes. Ten scenes may take a long time. If Kaggle disconnects, restart the notebook and update `VIDEO_WORKER_URL` and the token in Railway.